# 🔬 Module 18 — Statistiques inférentielles
> **Bootcamp Data Analyst — From Zero to Hero** | Niveau Intermédiaire · Module 18

---

## 🎯 Ce que tu seras capable de faire à la fin de ce module

- Expliquer à quoi servent les tests statistiques, en général
- Distinguer un test **paramétrique** d'un test **non paramétrique**, et savoir lequel choisir
- Formuler une hypothèse nulle (H0) et une hypothèse alternative (H1)
- Interpréter correctement une **p-value** — et éviter le piège d'interprétation le plus commun
- Comparer deux moyennes avec le **test t de Student** (classique) ou de **Welch** (plus robuste), vérifier l'égalité des variances (test de Levene), ou utiliser un **test de Mann-Whitney** (non paramétrique)
- Comparer **plus de deux moyennes** avec une **ANOVA**, et savoir pourquoi un test post-hoc (Tukey) est indispensable derrière
- Lire un **intervalle de confiance**
- Tester une association entre deux variables catégorielles avec un **test du χ² (chi-carré)**
- Distinguer signification **statistique** et signification **pratique/métier**

> ⚠️ Ce module ne demande aucun calcul à la main — l'angle est **l'interprétation**. En tant que DA, ton rôle n'est pas de dériver les formules, mais de savoir choisir le bon test, lire son résultat, et l'expliquer correctement à un∙e décideur∙se.

---

> ⏱️ **Durée estimée** : 3 heures  
> 🛠️ **Outils** : Python (`scipy.stats`, `statsmodels`) · R (`t.test`, `car::leveneTest`, `wilcox.test`, `aov`, `chisq.test`)  
> 📌 **Prérequis** : Module 02 et Module 11 (Niveau Débutant — statistiques descriptives)

---
## 1. À quoi servent les tests statistiques

Au Module 02 et au Module 11, tu as calculé des moyennes, médianes, écarts-types — tu **décrivais** un échantillon. Un **test statistique** répond à une question différente : *est-ce que ce que j'observe dans mon échantillon me permet de conclure quelque chose sur la population entière, ou sur une vraie différence entre deux groupes — ou est-ce que ça pourrait n'être que le hasard de qui se trouve dans mon échantillon ?*

Concrètement, un test statistique sert à répondre à des questions comme :

- **Comparer une moyenne à une valeur de référence**, ou comparer les moyennes de deux (ou plusieurs) groupes — *"le salaire moyen diffère-t-il entre deux départements ?"*
- **Tester un lien entre deux variables catégorielles** — *"la ville est-elle liée au type de contrat ?"*
- **Tester une corrélation entre deux variables continues** — *"l'ancienneté est-elle liée au salaire ?"*

Dans chacun de ces cas, le test te donne une réponse chiffrée à la question *"cet écart/ce lien observé est-il assez marqué pour qu'on ne puisse pas l'attribuer au hasard ?"* — plutôt qu'une simple impression visuelle sur un graphique.

---
## 2. Tests paramétriques vs non paramétriques — choisir le bon test

Pour une même question ("est-ce que ces deux moyennes diffèrent ?"), il existe presque toujours **deux familles de tests** :

| | Tests **paramétriques** | Tests **non paramétriques** |
|---|---|---|
| Hypothèse sur les données | Suppose une distribution connue (généralement **normale**) | Ne suppose (presque) rien sur la distribution — souvent basé sur les **rangs**, pas les valeurs brutes |
| Quand les utiliser | Données à peu près normales, échantillon suffisant | Données très asymétriques, valeurs extrêmes, petits échantillons, données ordinales |
| Avantage | Plus puissant (détecte un effet plus facilement) si les hypothèses sont respectées | Plus robuste — fonctionne même si les hypothèses paramétriques sont violées |
| Inconvénient | Résultat peu fiable si l'hypothèse de distribution est fausse | Un peu moins puissant si les hypothèses paramétriques étaient en fait respectées |

### Le même besoin, deux tests possibles

| Ce que tu veux tester | Version paramétrique | Version non paramétrique |
|---|---|---|
| Comparer les moyennes de **2 groupes** | Test t (Student/Welch) | Test de **Mann-Whitney** (Wilcoxon rank-sum) |
| Comparer les moyennes de **3 groupes ou plus** | ANOVA | Test de **Kruskal-Wallis** |
| Corrélation entre 2 variables continues | Corrélation de **Pearson** | Corrélation de **Spearman** |
| Lien entre 2 variables catégorielles | *(pas de version paramétrique équivalente)* | Test du **χ²** |

> 💡 **Réflexe pratique** : par défaut, commence par la version paramétrique (test t, par exemple) — plus simple et plus puissante. Bascule vers la version non paramétrique si tu constates une distribution clairement anormale (beaucoup de valeurs extrêmes, distribution très asymétrique) ou si ton échantillon est petit (moins d'une trentaine d'observations par groupe) et visiblement non normal. Le test du χ² n'a pas d'équivalent paramétrique : dès que tu compares des **catégories** (pas des moyennes), c'est le test à utiliser.

Dans la suite de ce module, on illustre le test de comparaison de **moyennes** avec les deux versions (test t et Mann-Whitney), puis le test d'**association** entre catégories avec le χ².

---
## 3. Le scénario, les hypothèses (H0/H1), et la p-value

### Le scénario

Tu es Data Analyst chez **Bamba & Associés**. La DRH te pose une question sensible :

> *« On m'a signalé un possible écart de salaire entre les femmes et les hommes dans l'entreprise. Peux-tu vérifier si c'est un vrai problème structurel, ou juste du hasard sur nos effectifs actuels ? »*

### H0 et H1

Un test d'hypothèse compare toujours deux affirmations :

- **H0 (hypothèse nulle)** : "il n'y a pas de différence/lien" — le scénario par défaut, celui qu'on cherche à réfuter
- **H1 (hypothèse alternative)** : "il y a une différence/lien"

Pour notre question RH : **H0** = "le salaire moyen est le même chez les femmes et les hommes" ; **H1** = "le salaire moyen diffère selon le genre".

### La p-value — ce qu'elle veut dire vraiment

La **p-value** est la probabilité d'observer un écart **au moins aussi marqué** que celui mesuré dans ton échantillon, **si H0 était vraie** (s'il n'y avait vraiment aucune différence dans la population).

- p-value **petite** (généralement < 0,05) → l'écart observé serait très improbable si H0 était vraie → on **rejette H0**, l'écart est jugé "statistiquement significatif"
- p-value **grande** (≥ 0,05) → l'écart observé est plausible même sous H0 → on ne peut **pas rejeter H0** — attention, ça ne veut **pas** dire "H0 est vraie" !

> ⚠️ **Le piège le plus commun** : la p-value **n'est pas** "la probabilité que H0 soit vraie", et ce n'est pas non plus "la probabilité de se tromper en rejetant H0". C'est la probabilité d'observer les données (ou plus extrême) **en supposant H0 vraie**. La nuance est subtile mais réelle — si tu dis un jour à ton manager *"il y a 3 % de chances que l'hypothèse nulle soit vraie"*, tu commets exactement cette erreur d'interprétation. La bonne formulation : *"si le salaire moyen était réellement identique entre les deux groupes, on aurait moins de 5 % de chances d'observer un écart aussi marqué que celui-ci sur cet échantillon"*.

---
## 4. Comparer deux moyennes — le test t (paramétrique)

### Le jeu de données

On simule un extrait RH de 60 employés de Bamba & Associés (données fictives, générées pour l'exercice).

**Python :**
```python
import pandas as pd
import numpy as np

np.random.seed(7)

n = 60
genres = ["F"]*30 + ["H"]*30
departements = np.random.choice(["RH","Finance","Commercial","IT","Direction"], n, p=[0.2,0.2,0.25,0.2,0.15])
contrats = np.random.choice(["CDI","CDD","Stage"], n, p=[0.65,0.22,0.13])
villes = np.random.choice(["Abidjan","Bouaké","Yamoussoukro","San-Pédro"], n, p=[0.5,0.2,0.15,0.15])

dept_base = {"RH":400000,"Finance":500000,"Commercial":420000,"IT":480000,"Direction":900000}
salaire = []
for i in range(n):
    base = dept_base[departements[i]]
    ecart_genre = -85000 if genres[i] == "H" else 0
    bruit = np.random.normal(0, 55000)
    salaire.append(round(max(150000, base + ecart_genre + bruit) / 10000) * 10000)

df = pd.DataFrame({"genre": genres, "departement": departements, "contrat": contrats,
                    "ville": villes, "salaire": salaire})
```

### Comparer les deux groupes

```python
print(df.groupby("genre")["salaire"].agg(["mean", "std", "count"]).round(0))
```

Regarde l'écart entre les deux moyennes affichées, et les deux écarts-types.

### Le test t de Student — la version classique

Le **test t de Student** est le test de référence pour comparer deux moyennes. Il repose sur trois hypothèses : les observations sont indépendantes, la variable est à peu près **normalement distribuée** dans chaque groupe, et — c'est le point clé — les deux groupes ont **la même variance** (on parle d'**homoscédasticité**).

**Python (`scipy.stats.ttest_ind`, `equal_var=True`) :**
```python
from scipy import stats

f_salaire = df[df.genre == "F"]["salaire"]
h_salaire = df[df.genre == "H"]["salaire"]

resultat_student = stats.ttest_ind(f_salaire, h_salaire, equal_var=True)  # test t de Student
print(f"t = {resultat_student.statistic:.3f}, p = {resultat_student.pvalue:.5f}")
```

**R (`t.test`, `var.equal = TRUE`) :**
```r
resultat_student <- t.test(salaire ~ genre, data = df, var.equal = TRUE)
print(resultat_student)
```

### Vérifier l'hypothèse d'égalité des variances — test de Levene

Le test de Student n'est fiable que si l'hypothèse d'égalité des variances tient. Pour la vérifier plutôt que de la supposer, utilise le **test de Levene** (H0 : les variances sont égales).

**Python (`scipy.stats.levene`) :**
```python
lev_stat, lev_p = stats.levene(f_salaire, h_salaire)
print(f"Levene : statistique = {lev_stat:.3f}, p = {lev_p:.5f}")
```

**R (`car::leveneTest`, nécessite `install.packages("car")`) :**
```r
library(car)
leveneTest(salaire ~ genre, data = df)
```

- **p < 0,05** au test de Levene → les variances sont significativement différentes → **n'utilise pas le test de Student**, passe à la version de Welch ci-dessous.
- **p ≥ 0,05** → rien n'indique que les variances diffèrent → le test de Student est utilisable tel quel.

### Le test t de Welch — quand les variances diffèrent (ou par défaut, par prudence)

Le test de **Welch** est une variante du test t qui ne suppose **pas** l'égalité des variances — il ajuste automatiquement ses degrés de liberté pour rester valide même quand les deux groupes sont dispersés très différemment (ce qui est fréquent en pratique, comme souvent avec des données de salaire).

**Python (`equal_var=False`) :**
```python
resultat_welch = stats.ttest_ind(f_salaire, h_salaire, equal_var=False)  # test t de Welch
print(f"t = {resultat_welch.statistic:.3f}, p = {resultat_welch.pvalue:.5f}")
```

**R (Welch par défaut si tu omets `var.equal`) :**
```r
resultat_welch <- t.test(salaire ~ genre, data = df)
print(resultat_welch)
```

> 💡 **Réflexe pratique** : beaucoup de statisticien∙ne∙s recommandent aujourd'hui d'utiliser Welch **par défaut**, sans même passer par le test de Levene — Welch donne un résultat quasi identique à Student quand les variances sont effectivement égales, mais reste valide quand elles ne le sont pas. Student n'a donc plus vraiment d'avantage pratique aujourd'hui ; il reste important de le connaître car c'est encore le test "par défaut" dans de nombreux outils et manuels.

### Interpréter ta p-value

Quelle que soit la valeur que tu obtiens (Student ou Welch), la lecture suit toujours la même logique :

- **p < 0,05** → tu rejettes H0. Formulation correcte : *"Si le salaire moyen était réellement identique entre femmes et hommes, la probabilité d'observer un écart aussi marqué que celui mesuré sur nos 60 employés serait de p × 100 %. L'écart observé est statistiquement significatif."*
- **p ≥ 0,05** → tu ne rejettes pas H0. Formulation correcte : *"On ne détecte pas d'écart statistiquement significatif entre les deux groupes sur cet échantillon."* (jamais *"il n'y a pas d'écart"* — voir section 3)

### La version non paramétrique — test de Mann-Whitney

Si la distribution des salaires est clairement asymétrique (beaucoup de petits salaires, quelques très gros — la loi normale est une mauvaise approximation, peu importe l'égalité des variances), préfère la version non paramétrique. Elle compare les **rangs** des valeurs plutôt que les valeurs elles-mêmes.

**Python (`scipy.stats.mannwhitneyu`) :**
```python
resultat_mw = stats.mannwhitneyu(f_salaire, h_salaire)
print(f"U = {resultat_mw.statistic:.1f}, p = {resultat_mw.pvalue:.5f}")
```

**R (`wilcox.test`, le nom historique du même test) :**
```r
resultat_mw <- wilcox.test(salaire ~ genre, data = df)
print(resultat_mw)
```

Même logique d'interprétation de la p-value que pour le test t. Si Student, Welch et Mann-Whitney donnent la même conclusion, c'est un bon signe de robustesse de ton résultat.

---
## 5. ANOVA — comparer plus de deux moyennes

Le test t compare **2** groupes. Mais Bamba & Associés a **5 départements** (RH, Finance, Commercial, IT, Direction) — comment savoir si le salaire moyen varie selon le département ?

### Pourquoi ne pas juste faire des tests t deux-à-deux ?

Avec 5 groupes, il y a 10 paires possibles. Tu pourrais être tenté∙e de faire 10 tests t. **Ne fais jamais ça** : plus tu multiplies les tests, plus le risque de trouver un résultat "significatif" **par pur hasard** augmente (le "problème des comparaisons multiples"). Avec 10 tests indépendants à un seuil de 5 %, la probabilité qu'au moins un ressorte significatif par hasard peut dépasser 40 %, même si aucune vraie différence n'existe.

L'**ANOVA** (Analysis of Variance) résout ce problème : elle teste les 5 groupes **en une seule fois**.

- **H0** : les moyennes de salaire sont égales dans tous les départements
- **H1** : au moins un département a une moyenne différente des autres

### Le principe (intuition, pas la formule)

L'ANOVA compare deux sources de variation : la variance **entre** les groupes (les départements ont-ils des moyennes très différentes ?) et la variance **à l'intérieur** de chaque groupe (les salaires sont-ils déjà très dispersés au sein d'un même département ?). Le résultat est un ratio, la **statistique F** : plus la variance entre les groupes domine la variance interne, plus F est grand, et plus la p-value est petite.

### Exécuter l'ANOVA

**Python (`scipy.stats.f_oneway`) :**
```python
groupes = [grp["salaire"].values for nom, grp in df.groupby("departement")]
f_stat, p_val = stats.f_oneway(*groupes)
print(f"F = {f_stat:.3f}, p = {p_val:.5f}")
```

**R (`aov`) :**
```r
modele <- aov(salaire ~ departement, data = df)
summary(modele)
```

Même logique d'interprétation que les tests précédents : p < 0,05 → tu rejettes H0, au moins un département a une moyenne de salaire différente des autres.

> ⚠️ **Ce que l'ANOVA ne te dit PAS** : une ANOVA significative te dit seulement *"au moins un groupe diffère"* — jamais **lequel**. Pour le savoir, il faut un **test post-hoc**.

### Le test post-hoc — savoir lequel(s) départements diffèrent

**Python (`pairwise_tukeyhsd`, comparaisons de Tukey) :**
```python
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=df["salaire"], groups=df["departement"], alpha=0.05)
print(tukey)
```

**R (`TukeyHSD`) :**
```r
TukeyHSD(modele)
```

Le test de Tukey compare **chaque paire de départements**, mais en ajustant les seuils de significativité pour contrôler le risque de faux positifs — exactement le problème des comparaisons multiples évité en amont par l'ANOVA. Le résultat affiche, pour chaque paire, un écart de moyennes, une p-value ajustée, et une colonne `reject` (`True`/`TRUE` si l'écart est significatif pour cette paire précise).

> 🔗 **Retiens la logique en 2 temps** : **ANOVA d'abord** (y a-t-il une différence quelque part parmi les groupes ?), **Tukey ensuite, seulement si l'ANOVA est significative** (entre quels groupes précisément ?). Sauter l'ANOVA et foncer directement sur des comparaisons deux-à-deux est l'erreur la plus fréquente — et celle qui gonfle artificiellement le nombre de "découvertes" statistiquement significatives.

### La version non paramétrique — Kruskal-Wallis

Si la distribution des salaires par département est clairement non normale, utilise l'équivalent non paramétrique vu en section 2 :

**Python :**
```python
kw_stat, kw_p = stats.kruskal(*groupes)
print(f"H = {kw_stat:.3f}, p = {kw_p:.5f}")
```

**R :**
```r
kruskal.test(salaire ~ departement, data = df)
```

Même limite que l'ANOVA : un résultat significatif indique qu'au moins un groupe diffère, sans dire lequel — le post-hoc adapté à Kruskal-Wallis (test de Dunn) suit la même logique que Tukey, mais dépasse le cadre de ce module.

---
## 6. L'intervalle de confiance — aller au-delà du "significatif ou pas"

Une p-value te dit "significatif ou pas", mais pas **l'ampleur** de l'écart. L'**intervalle de confiance (IC)** complète l'information — reprends le résultat du test de Welch (`resultat_welch`) calculé en section 4.

**Python :**
```python
ic = resultat_welch.confidence_interval(confidence_level=0.95)
print(f"IC 95% : [{ic.low:.0f}, {ic.high:.0f}] FCFA")
```

**R :**
```r
resultat_welch$conf.int
```

**Interprétation** : le résultat te donne une fourchette — *"on estime, avec 95 % de confiance, que l'écart réel de salaire moyen se situe entre [borne basse] et [borne haute] FCFA"*. C'est une information bien plus utile pour la DRH qu'un simple "c'est significatif" — elle donne une idée de l'**ampleur** du problème à corriger, pas juste de son existence.

---
## 7. Tester un lien entre deux catégories — test du χ²

Deuxième question RH : *« Y a-t-il un lien entre la ville où travaille un∙e employé∙e et le type de contrat qu'il/elle a ? »* Ici, les deux variables sont **catégorielles** (pas des moyennes à comparer) — comme vu en section 2, c'est le test du **χ² (chi-carré) d'indépendance** qu'il faut utiliser (il n'a pas de version "paramétrique" alternative).

### Construire le tableau de contingence

**Python :**
```python
df["abidjan"] = df["ville"].apply(lambda v: "Abidjan" if v == "Abidjan" else "Autre ville")
df["type_contrat"] = df["contrat"].apply(lambda c: "CDI" if c == "CDI" else "Autre contrat")

table = pd.crosstab(df["abidjan"], df["type_contrat"])
print(table)
```

### Le test

```python
chi2, p_val, dof, attendus = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.3f}, p = {p_val:.4f}, degrés de liberté = {dof}")
print("Effectifs attendus si aucun lien :")
print(attendus)
```

**R (`chisq.test`) :**
```r
table_r <- table(df$abidjan, df$type_contrat)
resultat_chi2 <- chisq.test(table_r)
print(resultat_chi2)
```

### Interpréter ta p-value

Même logique que pour le test t :

- **p < 0,05** → tu rejettes H0 : *"On détecte un lien statistiquement significatif entre la ville et le type de contrat."*
- **p ≥ 0,05** → tu ne rejettes pas H0 : *"On ne détecte pas de lien statistiquement significatif entre la ville et le type de contrat sur cet échantillon."*

> ⚠️ **Attention à la formulation** — ne dis jamais *"il n'y a pas de lien"* de façon catégorique. Un résultat non significatif peut vouloir dire deux choses très différentes : soit il n'y a vraiment aucun lien, soit l'échantillon est trop petit pour le détecter. Le test ne permet pas de trancher entre les deux.

### Une vérification avant de faire confiance au résultat

Le test du χ² n'est fiable que si les **effectifs attendus** (calculés sous H0, la variable `attendus` dans le code ci-dessus) sont suffisamment grands — la règle courante est **au moins 5 par case**. Vérifie ce point avant d'interpréter le résultat. Avec un tableau de contingence qui a des cases à 1 ou 2 individus attendus, le résultat du χ² devient peu fiable — il faut alors regrouper des catégories ou utiliser un test alternatif (test exact de Fisher).

---
## 8. Signification statistique ≠ importance pratique

Un dernier réflexe à avoir : un résultat peut être **statistiquement significatif** (p très petite) sans être **important en pratique**, et inversement.

- Avec un très grand échantillon (des millions de lignes), une différence minuscule et sans intérêt métier (ex : 500 FCFA d'écart de salaire) peut ressortir "statistiquement significative" simplement parce que l'échantillon est énorme.
- À l'inverse, un écart réel et important peut ne pas ressortir significatif si l'échantillon est trop petit — c'est le risque qu'on a signalé pour le test du χ² ci-dessus.

> 💡 **Réflexe DA** : toujours regarder la p-value **et** l'ampleur de l'effet (l'intervalle de confiance) **et** la taille de l'échantillon, avant de conclure quoi que ce soit à un∙e décideur∙se. La p-value seule ne raconte jamais toute l'histoire.

---
## ✅ Résumé du module

| Concept | Ce qu'il faut retenir |
|---|---|
| **Rôle d'un test statistique** | Trancher si un écart/lien observé est assez marqué pour ne pas être dû au hasard |
| **Paramétrique vs non paramétrique** | Paramétrique suppose une distribution (souvent normale) et est plus puissant si l'hypothèse tient ; non paramétrique (basé sur les rangs) est plus robuste mais un peu moins puissant |
| **H0 / H1** | H0 = "pas de différence/lien" (scénario par défaut) ; H1 = "il y a une différence/lien" |
| **p-value** | Probabilité d'observer un écart au moins aussi marqué **si H0 était vraie** — pas "la probabilité que H0 soit vraie" |
| **Seuil 0,05** | Convention courante : p < 0,05 → on rejette H0 ("statistiquement significatif") |
| **Test t de Student (`ttest_ind, equal_var=True` / `t.test(..., var.equal=TRUE)`)** | Version classique de comparaison de 2 moyennes — suppose des variances égales entre les groupes |
| **Test de Levene (`levene` / `car::leveneTest`)** | Vérifie l'hypothèse d'égalité des variances avant de choisir Student ou Welch |
| **Test t de Welch (`equal_var=False` / valeur par défaut de `t.test`)** | Ne suppose pas l'égalité des variances — recommandé par défaut en pratique |
| **Mann-Whitney (`mannwhitneyu` / `wilcox.test`)** | Version non paramétrique de comparaison de 2 moyennes — basée sur les rangs |
| **ANOVA (`f_oneway` / `aov`)** | Compare plus de 2 moyennes en une seule fois — évite le problème des comparaisons multiples de tests t répétés |
| **Test post-hoc (Tukey HSD)** | Indispensable après une ANOVA significative pour savoir **lequel(s)** des groupes diffèrent — l'ANOVA seule ne le dit pas |
| **Kruskal-Wallis (`kruskal` / `kruskal.test`)** | Version non paramétrique de l'ANOVA |
| **Intervalle de confiance** | Donne une fourchette plausible pour l'écart réel — complète la p-value |
| **Test du χ² (`chi2_contingency` / `chisq.test`)** | Teste un lien entre deux variables catégorielles, via un tableau de contingence — pas de version paramétrique équivalente |
| **Effectifs attendus ≥ 5** | Condition de validité du test du χ² — sinon regrouper les catégories ou utiliser un test exact |
| **"Non significatif" ≠ "pas de lien"** | Un résultat non significatif peut aussi venir d'un échantillon trop petit |
| **Significativité statistique ≠ importance pratique** | Toujours regarder p-value + ampleur de l'effet + taille d'échantillon ensemble |

---

## 🧠 Quiz — Vérifie ta compréhension

---

**Q1.** Tes données de salaire ont une distribution très asymétrique, avec quelques très gros salaires qui tirent la moyenne vers le haut. Quel test privilégier pour comparer deux groupes ?
- a) Le test t, toujours plus puissant
- b) Le test de Mann-Whitney, qui ne suppose pas une distribution normale
- c) Le test du χ², qui fonctionne avec n'importe quelle distribution

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — Le test de Mann-Whitney (non paramétrique, basé sur les rangs) est plus robuste face à une distribution asymétrique ou avec des valeurs extrêmes. Le test t (a) suppose une distribution à peu près normale — son résultat devient peu fiable si cette hypothèse est clairement violée. Le χ² (c) sert à tester un lien entre catégories, pas à comparer des moyennes.
</details>

---

**Q2.** Que signifie une p-value de 0,03 pour un test comparant deux moyennes ?
- a) Il y a 3 % de chances que l'hypothèse nulle soit vraie
- b) Si l'hypothèse nulle était vraie (pas de vraie différence), on aurait 3 % de chances d'observer un écart au moins aussi marqué que celui mesuré
- c) L'écart mesuré a 97 % de chances d'être correct

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — C'est la définition exacte de la p-value : une probabilité conditionnelle **sous H0**, pas une probabilité sur H0 elle-même. Les options a) et c) sont l'erreur d'interprétation la plus fréquente, à ne jamais reproduire devant un∙e décideur∙se.
</details>

---

**Q3.** Ton test t donne p = 0,45. Quelle formulation est correcte ?
- a) "On ne détecte pas d'écart statistiquement significatif entre les deux groupes sur cet échantillon"
- b) "Il n'y a aucune différence entre les deux groupes"
- c) "L'hypothèse nulle est vraie à 55 %"

<details>
<summary>👉 Voir la réponse</summary>

✅ **a)** — Un résultat non significatif ne prouve pas l'absence de différence (b) — elle peut exister mais ne pas être détectée (échantillon trop petit, bruit trop important). c) répète l'erreur d'interprétation de la p-value vue au Q2.
</details>

---

**Q4.** Ton tableau de contingence pour un test du χ² a des cases avec seulement 1 ou 2 individus attendus. Que dois-tu faire ?
- a) Ignorer le problème, le test du χ² fonctionne toujours
- b) Regrouper des catégories pour augmenter les effectifs par case, ou utiliser un test exact de Fisher
- c) Multiplier chaque effectif par 5

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — Le test du χ² n'est fiable que si les effectifs attendus sont suffisants (règle courante : ≥ 5 par case). En dessous, il faut regrouper des catégories ou utiliser une alternative comme le test exact de Fisher, plus adapté aux petits effectifs.
</details>

---

**Q5.** Ton ANOVA comparant le salaire moyen entre 5 départements donne p = 0,001. Que peux-tu conclure ?
- a) Tous les départements ont des salaires moyens différents les uns des autres
- b) Au moins un département a un salaire moyen différent des autres — mais tu ne sais pas encore lequel
- c) Le département avec le salaire le plus élevé est significativement différent de tous les autres

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — Une ANOVA significative te dit seulement qu'il existe **au moins une** différence quelque part parmi les groupes. Pour savoir précisément entre quelles paires de départements l'écart est significatif, il faut un test post-hoc (Tukey HSD) — sans lui, a) et c) sont des conclusions que tes données ne permettent pas encore de tirer.
</details>

---

**Q6.** Ton test de Levene comparant les variances de salaire entre femmes et hommes donne p = 0,02. Que dois-tu en conclure pour choisir ton test t ?
- a) Les variances sont significativement différentes — utilise le test de Welch, pas Student
- b) Les variances sont égales — utilise le test de Student
- c) Le test de Levene ne sert à rien pour choisir entre Student et Welch

<details>
<summary>👉 Voir la réponse</summary>

✅ **a)** — p < 0,05 au test de Levene signifie qu'on rejette l'hypothèse d'égalité des variances (H0 du test de Levene). Le test de Student suppose justement cette égalité — il faut donc utiliser le test de Welch, qui reste valide même quand les variances diffèrent. En cas de doute, Welch par défaut reste le choix le plus sûr.
</details>

---

## ➡️ Module suivant

Tu sais maintenant choisir un test, l'exécuter et interpréter son résultat correctement. Dans le **Module 19**, on passe à l'automatisation Python — consommer une API et écrire un script réutilisable.

> **→ Module 19 — Automatisation Python (API & scripts)**